## Data Collection & Preperation

In [1]:
# Imports
import pandas as pd
import numpy as np
from nhlpy import NHLClient # https://pypi.org/project/nhl-api-py/#game-center
client = NHLClient()

from nhlpy.api.query.builder import QueryBuilder
from nhlpy.api.query.filters.season import SeasonQuery
from nhlpy.api.query.filters.game_type import GameTypeQuery

In [2]:
# Get all season info
season_info = client.misc.season_specific_rules_and_info()
season_info = pd.DataFrame(season_info)

In [3]:
def time_to_seconds(t):
    mins, secs = t.split(":")
    return int(mins) * 60 + int(secs)

In [4]:
shot_data = []

seasons = ["20212022", "20222023", "20232024", "20242025"]

season_prefix = {
    "20212022": "202102",
    "20222023": "202202",
    "20232024": "202302",
    "20242025": "202402",
}

for season in seasons:
    print(f"Season: {season}")
    # get num games
    specific_season = season_info[season_info["id"].astype(str) == season]
    num_reg_games = specific_season['totalRegularSeasonGames'].values[0]

    # get skater and goalie stats for season
    filters = [
        SeasonQuery(season_start=season, season_end=season),
        GameTypeQuery(game_type="2"),
    ]
    context = QueryBuilder().build(filters=filters)

    # skaters
    all_skaters = []
    start = 0
    while True:
        response = client.stats.skater_stats_with_query_context(
            report_type='summary',
            query_context=context,
            aggregate=True,
            limit=100,
            start=start
        )
        all_skaters.extend(response['data'])
        if len(all_skaters) >= response['total']:
            break
        start += 100

    shooter_stats_df = pd.DataFrame(all_skaters)
    shooter_stats_df = shooter_stats_df.dropna(subset=['playerId'])
    shooter_stats_df['playerId'] = shooter_stats_df['playerId'].astype(np.float64)
    print(f"Skaters: {len(shooter_stats_df)}")

    # goalies
    all_goalies = []
    start = 0
    while True:
        response = client.stats.goalie_stats_summary(
            start_season=season,
            end_season=season,
            limit=100,
            start=start
        )
        if isinstance(response, list):
            all_goalies.extend(response)
            if len(response) < 100:
                break
            start += 100
        else:
            all_goalies.extend(response['data'])
            if len(all_goalies) >= response['total']:
                break
            start += 100

    goalie_stats_df = pd.DataFrame(all_goalies)
    goalie_stats_df = goalie_stats_df.dropna(subset=['playerId'])
    goalie_stats_df['playerId'] = goalie_stats_df['playerId'].astype(np.float64)
    print(f"Goalies: {len(goalie_stats_df)}")

    # get averages for missing values
    league_avg_shooting_pct = shooter_stats_df['shootingPct'].dropna().mean()
    league_avg_save_pct = goalie_stats_df['savePct'].dropna().mean()

    # Loop through all games
    for i in range(1, num_reg_games+1): # game loop
        data = client.game_center.play_by_play(game_id = f"{season_prefix[season]}{i:04d}")
        home_team_id = data['homeTeam']['id']

        home_goals = 0
        away_goals = 0

        for j, play in enumerate(data["plays"]): # play loop
            play_details = play.get("details", {})

            # details
            shot_type = play_details.get("shotType")

            x = play_details.get("xCoord")
            y = play_details.get("yCoord")

            time = play.get("timeInPeriod")

            situation_code = play.get("situationCode") # situationCode = [Away Goalie count][Away Player count][Home Player count][Home Goalie count]


            if shot_type is not None and x is not None and y is not None and time is not None and situation_code:
                # SKIP SHOOTOUT
                if play['periodDescriptor'].get("periodType") == 'SO':
                    continue

                # skater differential
                shooter_is_home = (home_team_id == play_details.get("eventOwnerTeamId"))

                attackers = int(str(situation_code)[2]) if shooter_is_home else int(str(situation_code)[1])
                defenders = int(str(situation_code)[1]) if shooter_is_home else int(str(situation_code)[2])
                skater_dif = attackers - defenders

                shooting_on_empty = (int(str(situation_code)[0]) == 0) if shooter_is_home else (int(str(situation_code)[3]) == 0)

                # coordinates
                x_norm = abs(x)
                y_norm = y if x > 0 else -y

                # https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcS1CE8zZckRo6CVt0U-HZJhJMXWVGJZA6ZaYQ&s

                distance = np.sqrt((89 - x_norm)**2 + y_norm**2)
                angle = np.degrees(np.arctan2(abs(y_norm), 89 - x_norm))

                # time, period, OT
                time = time_to_seconds(time)
                period = play['periodDescriptor']['number']
                overtime = (period == 4)


                # Rebound
                rebound = False
                last_play = data['plays'][j-1] if (j>0) else False
                last_play_details = last_play.get("details", {})
                if last_play and last_play_details.get("shotType") and last_play.get("timeInPeriod"):
                    time_last = time_to_seconds(last_play.get("timeInPeriod"))
                    if ((time - time_last) <= 3):
                        rebound = True

                # Goalie save pct
                goalie_id = np.float64(play_details.get("goalieInNetId"))
                goalie_row = goalie_stats_df[goalie_stats_df['playerId'] == goalie_id]
                goalie_save_pct = goalie_row['savePct'].values[0] if len(goalie_row) > 0 else league_avg_save_pct

                # Shooting percentage
                shooter_id = play_details.get("shootingPlayerId") or play_details.get("scoringPlayerId")
                shooter_id = np.float64(shooter_id) if shooter_id else np.nan
                shooter_row = shooter_stats_df[shooter_stats_df['playerId'] == shooter_id]
                shooting_pct = shooter_row['shootingPct'].values[0] if len(shooter_row) > 0 else league_avg_shooting_pct

                # goal
                goal = (play.get("typeDescKey") == "goal")

                team_score = home_goals if shooter_is_home else away_goals
                opponent_score = away_goals if shooter_is_home else home_goals
                goal_dif = team_score - opponent_score
                
                # add values to shot_data
                shot_data.append({
                    'season': season,
                    'shot_type': shot_type,
                    'distance': distance,
                    'angle': angle,
                    'rebound': rebound,
                    'skater_dif': skater_dif,
                    'shooting_on_empty': shooting_on_empty,
                    'period': period,
                    'overtime': overtime,
                    'goal_dif': goal_dif,
                    'goalie_save_pct': goalie_save_pct,
                    'shooting_pct': shooting_pct,
                    'goal': goal,
                })

            if play.get('typeDescKey') == 'goal':
                if play.get('details', {}).get('eventOwnerTeamId') == home_team_id:
                    home_goals += 1
                else:
                    away_goals += 1


        if i % 100 == 0:
            print(f"Game {i}/{num_reg_games} \n Total Shots: {len(shot_data)}")

Season: 20212022
Skaters: 1004
Goalies: 119
Game 100/1312 
 Total Shots: 8529
Game 200/1312 
 Total Shots: 17024
Game 300/1312 
 Total Shots: 25594
Game 400/1312 
 Total Shots: 34196
Game 500/1312 
 Total Shots: 42739
Game 600/1312 
 Total Shots: 51389
Game 700/1312 
 Total Shots: 59915
Game 800/1312 
 Total Shots: 68515
Game 900/1312 
 Total Shots: 77252
Game 1000/1312 
 Total Shots: 86032
Game 1100/1312 
 Total Shots: 94806
Game 1200/1312 
 Total Shots: 103527
Game 1300/1312 
 Total Shots: 112247
Season: 20222023
Skaters: 951
Goalies: 107
Game 100/1312 
 Total Shots: 121944
Game 200/1312 
 Total Shots: 130795
Game 300/1312 
 Total Shots: 139411
Game 400/1312 
 Total Shots: 148057
Game 500/1312 
 Total Shots: 156556
Game 600/1312 
 Total Shots: 165324
Game 700/1312 
 Total Shots: 174023
Game 800/1312 
 Total Shots: 182901
Game 900/1312 
 Total Shots: 191739
Game 1000/1312 
 Total Shots: 200462
Game 1100/1312 
 Total Shots: 209080
Game 1200/1312 
 Total Shots: 217704
Game 1300/1312 
 T

In [5]:
shot_data = pd.DataFrame(shot_data)
shot_data.head()

,season,shot_type,distance,angle,rebound,skater_dif,shooting_on_empty,period,overtime,goal_dif,goalie_save_pct,shooting_pct,goal
0,20212022,wrist,42.520583,48.814075,False,0,False,1,False,0,0.91934,0.17427,False
1,20212022,wrist,30.610456,38.367485,False,0,False,1,False,0,0.91648,0.12121,False
2,20212022,wrist,85.381497,18.434949,False,0,False,1,False,0,0.91934,0.03488,False
3,20212022,wrist,29.274562,7.853313,False,0,False,1,False,0,0.91648,0.13043,False
4,20212022,wrist,26.305893,8.746162,False,0,False,1,False,0,0.91934,0.12587,False


In [6]:
# save to csv
shot_data.to_csv('shot_data.csv', index=False)